[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/LevantamentoLidarIME/blob/main/01_Intro.ipynb)

# 01 — Introdução ao LiDAR e às nuvens de pontos 3D

**Instituto Militar de Engenharia — 2º ano do Curso Básico — 2º semestre de 2026**

Este notebook apresenta os fundamentos necessários para compreender um levantamento com o **SLAM 100** e realiza duas práticas: leitura do `bunny.pcd` com Open3D e inspeção de um arquivo LAS público.

## Objetivos de aprendizagem

Ao final, o aluno deverá ser capaz de:

1. explicar como um LiDAR estima distância e suas aplicações;
2. interpretar coordenadas, intensidade, retornos, cor e classificação;
3. explicar por que SLAM, georreferenciamento e controle independente são etapas distintas;
5. reconhecer as finalidades dos formatos PCD, LAS/LAZ, E57 e PLY;
6. baixar, validar, ler e visualizar nuvens de pontos em um notebook.

## 1. O que é LiDAR?

**LiDAR** (*Light Detection and Ranging*) é uma técnica ativa de sensoriamento: o instrumento emite luz laser, detecta a energia que retorna e estima a distância até uma superfície. Em um sistema de tempo de voo, a relação idealizada é

$$R = \frac{c\,\Delta t}{2}$$

em que $R$ é o alcance, $c$ é a velocidade da luz e $\Delta t$ é o intervalo entre emissão e recepção. A divisão por dois representa o percurso de ida e volta. Alguns sensores usam diferença de fase ou outras estratégias, mas o produto geométrico continua dependendo da combinação entre distância e direção do feixe.

Conhecidos a posição e a orientação do sensor, uma medição polar pode ser transformada em coordenadas cartesianas. Em uma convenção simples:

$$x=R\cos(\theta)\cos(\phi),\quad y=R\cos(\theta)\sin(\phi),\quad z=R\sin(\theta)$$

<center><img src="media/imgs/LidarPortatil.jpg"> </center>

<center><img src="media/imgs/LidarCarro.jpg"> </center>


<center><img src="media/imgs/LidarDrone.jpg"> </center>


#### Como o LiDAR "enxerga" o mundo

<center><img src="media/imgs/lidar-view-from-car.jpg"> </center>


### 2. Nuvem de pontos e seus atributos

Uma nuvem é um conjunto de retornos discretos (pontos). Cada ponto possui ao menos as coordenadas **X, Y, Z** e pode carregar: tempo GPS, intensidade, número do retorno, quantidade de retornos, ângulo de varredura, classe, cor RGB, identificação da linha/faixa e indicadores de qualidade.


### 1.2 Precisão, acurácia e fontes de erro

- **Precisão:** repetibilidade/espalhamento das medições.
- **Acurácia:** proximidade entre o resultado e uma referência aceita.
- **Erro relativo:** desalinhamento interno, como uma parede duplicada entre passagens.
- **Erro absoluto:** deslocamento ou rotação do conjunto em relação ao referencial do projeto.

Principais fontes: ruído de alcance, incidência oblíqua, superfícies escuras, brilhantes ou transparentes, alvos móveis, oclusão, calibração, sincronização, erro de trajetória, deriva do SLAM e transformação geodésica. A especificação do equipamento não substitui o controle da nuvem final.

## 2. LiDAR móvel e SLAM

No mapeamento móvel, o scanner muda de posição enquanto mede. O **SLAM** (*Simultaneous Localization and Mapping*) estima simultaneamente a trajetória do sistema e um mapa consistente, associando observações repetidas ao longo do percurso. IMU, câmera e GNSS podem auxiliar, conforme o sistema.

Um fluxo conceitual é:

1. aquisição e sincronização das observações;
2. estimativa incremental do movimento;
3. associação/registro entre trechos;
4. detecção de fechamento de loop;
5. otimização global da trajetória;
6. transformação para o CRS do projeto usando apoio;
7. verificação independente da acurácia.

Fechar o loop reduz deriva, mas não garante georreferenciamento. Pontos usados no ajuste são **controle**; pontos reservados para avaliar o resultado são **verificação**.

### 2.1 Condições favoráveis e degeneradas

O SLAM tende a funcionar melhor com geometria variada, sobreposição, movimento suave e revisitas. Corredores longos e repetitivos, grandes planos sem feições, vidro, vegetação móvel, multidões, mudanças bruscas e poucas ligações entre setores podem degradar a solução.

Para o SLAM 100, registre sempre modelo/número de série, firmware, aplicativo, perfil de aquisição, acessórios, horário, operador e procedimento de inicialização. Valores de alcance, taxa e desempenho devem ser copiados da documentação correspondente à versão usada.

## 3. Sistemas de coordenadas e formatos

Uma nuvem pode estar em coordenadas **locais**, no referencial do sensor ou em um **CRS geodésico/projetado**. Para uso métrico, documente datum/realização e época, projeção e zona, unidade, eixo, altitude elipsoidal ou ortométrica e modelo geoidal. Um arquivo sem CRS não permite inferir corretamente onde nem em que referencial os pontos se encontram.

| Formato | Uso típico | Observações |
|---|---|---|
| PCD | PCL, robótica e experimentos | Estrutura simples; pode ser ASCII ou binária |
| LAS | Intercâmbio geoespacial LiDAR | Cabeçalho, pontos, atributos, VLR/EVLR |
| LAZ | LAS comprimido sem perda | Menor armazenamento; requer codec compatível |
| E57 | Varreduras 3D e intercâmbio | Pode armazenar poses e imagens |
| PLY | Geometria, cor e malhas | Útil para visualização e processamento 3D |

A especificação LAS é mantida pela ASPRS. Nem todo LAS contém CRS ou classificação válida: sempre inspecione o cabeçalho e os metadados.

## 4. Preparação do ambiente

Instale o ambiente do repositório com `python -m pip install -r requirements.txt`. No Colab, execute a célula abaixo uma vez, removendo o comentário. Reinicie o kernel se solicitado.

In [ ]:
# Execute somente se as bibliotecas ainda não estiverem instaladas:
# %pip install -q open3d laspy[lazrs] plotly pandas

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
import hashlib
import numpy as np
import pandas as pd
from IPython.display import display
import open3d as o3d
import laspy

PASTA_DADOS = Path('dados/amostras')
PASTA_DADOS.mkdir(parents=True, exist_ok=True)
print('Open3D:', o3d.__version__, '| laspy:', laspy.__version__)

## 5. Download reprodutível dos dados

As URLs estão fixadas em versões dos repositórios oficiais. O SHA-256 impede o uso silencioso de um arquivo incompleto ou diferente. Se a cópia local já for válida, não há novo download.

- `bunny.pcd`: amostra da **Point Cloud Library**, versão `pcl-1.8.0`;
- `arquivo.las`: `1.2-with-color.las`, teste público do **PDAL 2.9.2**.

> Essas amostras ensinam formatos e visualização; não representam uma campanha do SLAM 100.

In [ ]:
AMOSTRAS = {
    'bunny.pcd': {
        'url': 'https://raw.githubusercontent.com/PointCloudLibrary/pcl/pcl-1.8.0/test/bunny.pcd',
        'sha256': 'f5418a140de7a21adc8a46b83bca8ee05b0ebd5a5fe3ca8a0925bcb6535ed511',
    },
    'arquivo.las': {
        'url': 'https://raw.githubusercontent.com/PDAL/PDAL/2.9.2/test/data/las/1.2-with-color.las',
        'sha256': '1d3e00eae856bffa6e55778dbfc05bdedc992504c64dbf366d5fd2eb6489ee45',
    },
}

def sha256_arquivo(caminho):
    digest = hashlib.sha256()
    with caminho.open('rb') as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b''):
            digest.update(bloco)
    return digest.hexdigest()

def baixar_validar(nome, url, sha256_esperado):
    destino = PASTA_DADOS / nome
    if not destino.exists() or sha256_arquivo(destino) != sha256_esperado:
        requisicao = Request(url, headers={'User-Agent': 'LevantamentoLidarIME/1.0'})
        with urlopen(requisicao, timeout=60) as resposta:
            destino.write_bytes(resposta.read())
    sha256_obtido = sha256_arquivo(destino)
    if sha256_obtido != sha256_esperado:
        raise ValueError(f'Falha de integridade em {nome}: {sha256_obtido}')
    return destino

caminhos = {nome: baixar_validar(nome, **dados) for nome, dados in AMOSTRAS.items()}
pd.DataFrame([
    {'arquivo': nome, 'bytes': caminho.stat().st_size, 'sha256': sha256_arquivo(caminho)}
    for nome, caminho in caminhos.items()
] )

## 6. Prática 1 — `bunny.pcd` com Open3D

O PCD informa no cabeçalho campos, tipos, largura, altura, número de pontos e codificação. O exemplo possui somente `x y z`, 397 pontos e dados ASCII. `read_point_cloud` converte o arquivo para `open3d.geometry.PointCloud`.

In [ ]:
bunny = o3d.io.read_point_cloud(str(caminhos['bunny.pcd']))
if bunny.is_empty():
    raise ValueError('O arquivo bunny.pcd foi lido, mas não contém pontos.')

xyz_bunny = np.asarray(bunny.points)
resumo_bunny = pd.DataFrame({
    'mínimo': xyz_bunny.min(axis=0),
    'máximo': xyz_bunny.max(axis=0),
    'amplitude': np.ptp(xyz_bunny, axis=0),
}, index=['X', 'Y', 'Z'])
print(bunny)
display(resumo_bunny.round(5))

In [ ]:
# Visualizador Plotly do Open3D: interativo e incorporado à saída do notebook.
o3d.visualization.draw_plotly(
    [bunny],
    window_name='Bunny — PCD lido com Open3D',
    width=800,
    height=550,
    point_sample_factor=1.0,
    front=[0.0, 0.0, 1.0],
    up=[0.0, 1.0, 0.0],
    zoom=0.7,
)

## 7. Prática 2 — arquivo LAS público

O LAS armazena coordenadas escaladas e atributos em registros binários. `laspy` aplica automaticamente escala e *offset* ao expor `las.x`, `las.y` e `las.z`. Primeiro inspecionaremos cabeçalho, limites e dimensões; depois criaremos uma geometria Open3D.

In [ ]:
las = laspy.read(caminhos['arquivo.las'])
crs = las.header.parse_crs()
dimensoes = list(las.point_format.dimension_names)

resumo_las = pd.Series({
    'versão LAS': str(las.header.version),
    'formato do ponto': las.header.point_format.id,
    'quantidade de pontos': las.header.point_count,
    'escalas XYZ': tuple(las.header.scales),
    'offsets XYZ': tuple(las.header.offsets),
    'CRS declarado': crs.to_string() if crs else 'não informado no arquivo',
    'dimensões': ', '.join(dimensoes),
})
resumo_las.to_frame('valor')

In [ ]:
xyz_las = np.column_stack((las.x, las.y, las.z))
centro_las = xyz_las.mean(axis=0)

# Centralizar apenas para estabilidade da visualização; as coordenadas originais permanecem em xyz_las.
nuvem_las = o3d.geometry.PointCloud()
nuvem_las.points = o3d.utility.Vector3dVector(xyz_las - centro_las)

if {'red', 'green', 'blue'}.issubset(dimensoes):
    rgb = np.column_stack((las.red, las.green, las.blue)).astype(float)
    divisor_rgb = 255.0 if rgb.max() <= 255 else 65535.0
    nuvem_las.colors = o3d.utility.Vector3dVector(np.clip(rgb / divisor_rgb, 0, 1))

limites_las = pd.DataFrame({
    'mínimo': xyz_las.min(axis=0),
    'máximo': xyz_las.max(axis=0),
    'amplitude': np.ptp(xyz_las, axis=0),
}, index=['X', 'Y', 'Z'])
display(limites_las.round(3))
print(f'Centro removido somente da exibição: {centro_las.round(3)}')

In [ ]:
# A figura fica incorporada ao notebook; arraste para girar e use a roda para aproximar.
o3d.visualization.draw_plotly(
    [nuvem_las],
    window_name='Amostra LAS pública — PDAL',
    width=850,
    height=550,
    point_sample_factor=1.0,
    up=[0.0, 0.0, 1.0],
    zoom=0.7,
)

Pretendendo usar o scanner lidar da SE6 para obter uma nuvem de pontos das instalações da própria SE6. 
A finalidade é reconstrução 3D de ambientes e seus subprodutos.
Os ganhos são basicamente: velocidade, quantidade, precisão, multi-aplicação ao custo financeiro de se ter o aparelho
Por exemplo, para montar uma planta baixa das instalações, em vez de medir pontos específicos nas paredes, com a nuvem de pontos capturamos, paredes, chão, teto, móveis... depois separamos por suas respectivas classes e filtramos as paredes.
Um nível de detalhe bem maior sem muitas intervenções do operador o que diminui as chances de erros grosseiros.
O tempo de coleta, é basicamente uma caminhada ao redor da seção com o aparelho em mãos.
E o modelo 3D é navegável, extrapolando as possibilidades do CAD
O serviço que vamos fazer é levantamento interno de instalações.  Nada impede de um aluno juntar as idéias e bolar um negócio com emprego em interiores de florestas, navios, cavernas. Se acoplar no drone,  emprego no exterior de florestas, fazendas, campo de obra.
Fazer o levantamento significa, em outras palavras, fazer um catálogo de tudo que um determinado espaço tem.
O lidar em conjunto com fotos, é de longe, a forma mais rápida, precisa e completa.